### SPARK STREAMING

In [0]:
entities= ['customers','payments','locations','vehicles','drivers','trips']

In [0]:

for entity in entities:

    df_batch = spark.read\
        .format('csv')\
            .option('header',True)\
                .option('inferSchema',True)\
                    .load(f'/Volumes/pyspark_dbt/source/source_data/{entity}/{entity}.csv')
    df_batch.display()
    # /Volumes/pyspark_dbt/source/source_data/customers/customers.csv
    entitySchema = df_batch.schema   

    df = spark.readStream\
        .format('csv')\
            .option('header',True)\
                .schema(entitySchema)\
                    .load(f'/Volumes/pyspark_dbt/source/source_data/{entity}')

    df.writeStream\
        .format('delta')\
            .outputMode('append')\
                .option('checkpointLocation',f'/Volumes/pyspark_dbt/bronze/checkpoint/{entity}')\
                    .trigger(once=True)\
                        .toTable(f'pyspark_dbt.bronze.{entity}')

In [0]:
%sql
select * from pyspark_dbt.silver.vehicles